# 102 - Databricks Declarative Automation Bundles and GitHub CI/CD

Databricks Asset Bundles are now named **Declarative Automation Bundles**. Existing `databricks bundle ...` commands and `databricks.yml` files continue to use the word `bundle`. A bundle treats Databricks project code, resource definitions, permissions, environments, and deployment settings as source-controlled configuration.

This lesson deploys a deliberately small serverless notebook job. The workload is simple so the focus remains on CLI authentication, bundle configuration, GitHub collaboration, CI/CD, identity, deployment state, and promotion.

## Learning outcomes

- Explain what bundles manage and what they do not manage.
- Install and authenticate the Databricks CLI.
- Read and author `databricks.yml` and resource YAML.
- Use variables, substitutions, includes, sync rules, targets, permissions, and `run_as`.
- Validate, deploy, run, inspect, and intentionally destroy a bundle target.
- Use GitHub pull requests and Actions for validation and production deployment.
- Configure secretless GitHub-to-Databricks authentication with OIDC workload identity federation.

> Bundle commands run from a terminal or CI runner, not from Databricks notebook compute. The command cells are examples to copy into a local terminal.

## 1. Why bundles?

Without a bundle, teams often version notebook code but leave job configuration, schedules, permissions, compute, parameters, and pipeline settings as manual workspace state. A bundle places these definitions beside the code and gives them a repeatable lifecycle.

### Major features

| Feature | Purpose |
|---|---|
| Resource as code | Define jobs, Lakeflow pipelines, dashboards and other supported workspace resources |
| Source synchronization | Upload notebooks, Python, SQL, configuration, and artifacts required by resources |
| Targets | Isolate development, staging, and production settings |
| Variables/substitutions | Reuse configuration without hard-coded catalog, schema, identity, or path values |
| Deployment modes/presets | Apply safe development or production behavior such as name prefixes and paused triggers |
| Artifacts | Build and deploy wheels/JARs and reference them from resources |
| Permissions and `run_as` | Separate who deploys, who manages, and which identity executes |
| State tracking | Update resources owned by a specific bundle identity and target instead of duplicating them |
| Validation | Check configuration and resource schemas before deployment |
| CI/CD integration | Run the same validate/deploy/run lifecycle from GitHub Actions or another CI system |
| Templates | Standardize project layout, permissions, tests and CI across teams |
| Generate/bind | Adopt an existing workspace resource into bundle configuration |

Bundles deploy Databricks project resources; they are not a replacement for Terraform that provisions the account, networking, identity platform, storage, or the workspace itself. A common boundary is Terraform for platform infrastructure and bundles for application/data-product resources.

## 2. Architecture and lifecycle

```text
Developer branch -> pull request -> validate/test -> merge to main
                                                |
                                                v
GitHub Actions OIDC -> Databricks service principal -> bundle deploy -t prod
                                                |
                                                v
                         workspace files + job/pipeline configuration + state
```

Bundle lifecycle: **initialize or author -> validate -> deploy -> run -> observe -> update/redeploy**. `destroy` is an explicit cleanup action, not part of every deployment. Deployment and job execution are separate: `deploy` creates/updates the job definition; `run` triggers it.

## 3. Repository layout used in this lesson

The complete runnable companion project is in `day10/asset_bundle_example`.

```text
asset_bundle_example/
|-- databricks.yml
|-- resources/
|   `-- orders_job.yml
|-- src/
|   `-- orders_summary.py
|-- .github/workflows/
|   `-- databricks-bundle.yml
|-- .gitignore
`-- README.md
```

`databricks.yml` identifies the bundle and targets. Included resource YAML defines the job. The source notebook contains business logic. GitHub Actions owns the automated validation and promotion workflow.

## 4. Install and verify the Databricks CLI

Use a current Databricks CLI that supports bundles. Do not install the retired Python package named `databricks-cli`. Follow the official installation method for your operating system, then verify the executable being used.

In [ ]:
# Run in a local terminal (bash), not as notebook Python.
# databricks version
# which databricks
# databricks --help
# databricks bundle --help

### Windows PowerShell checks

```powershell
databricks version
Get-Command databricks
databricks bundle --help
```

If `bundle` is not recognized, an old CLI is earlier on `PATH`. Remove the obsolete installation or correct `PATH`; do not mix two CLIs.

## 5. Local authentication

For interactive development, use OAuth user-to-machine login. A named profile makes the selected workspace explicit. Avoid placing personal access tokens in YAML, shell history, notebooks, or Git.

In [ ]:
# Local terminal commands
# databricks auth login --host https://YOUR-WORKSPACE.cloud.databricks.com --profile training
# databricks auth profiles
# databricks current-user me --profile training
# databricks workspace list /Workspace/Users --profile training

### Authentication resolution

The CLI uses unified authentication. Authentication can come from an explicit `--profile`, bundle target configuration, environment variables, or CI workload identity. Make the source deliberate and verify it with `databricks current-user me` before deployment. A successful login does not imply permission to create jobs, write workspace files, use compute, or create Unity Catalog objects.

## 6. Create a bundle

You can start from an official template or author the small files directly. Templates are helpful for consistent project standards.

In [ ]:
# Local terminal
# mkdir simple-orders-bundle && cd simple-orders-bundle
# databricks bundle init default-python
#
# Or use the completed lesson project:
# cd day10/asset_bundle_example

## 7. Understand `databricks.yml`

The companion configuration uses these ideas:

- `bundle.name`: stable project identity.
- `include`: split resources into maintainable YAML files.
- `variables`: environment-dependent values with defaults or overrides.
- `workspace.root_path`: deployed files and bundle state location.
- `sync`: files included/excluded from workspace synchronization.
- `targets.dev`: default development target with user isolation.
- `targets.prod`: production mode, shared path, explicit `run_as`, and permissions.

Common substitutions include `${bundle.name}`, `${bundle.target}`, `${workspace.current_user.userName}`, `${workspace.current_user.short_name}`, `${var.catalog}`, and `${resources.jobs.orders_summary_job.id}`. Substitutions are resolved by the bundle engine; shell environment variables use different syntax.

In [ ]:
# databricks.yml - simplified excerpt
bundle:
  name: simple-orders-bundle
include:
  - resources/*.yml
variables:
  catalog:
    default: workspace
targets:
  dev:
    default: true
    mode: development
  prod:
    mode: production

### Development versus production modes

Development mode normally prefixes resource names with the current user, adds development tags, pauses schedules/triggers, and enables safe personal isolation. Production mode applies stricter validation and should use an explicit run identity and permissions. Modes are useful defaults, not a complete security boundary. Review generated configuration with `bundle validate` and `bundle summary`.

Use target presets when you need organization-specific behavior such as name prefixes, tags, maximum concurrent runs, paused triggers, or pipeline development settings.

## 8. Resource definition: a small serverless job

The resource key `orders_summary_job` is the logical identifier used by bundle commands. The displayed job name can vary by target. `notebook_path` is relative to the resource YAML file, and parameters use bundle variables. No cloud-specific node type is required because the example uses a serverless environment.

In [ ]:
# resources/orders_job.yml
resources:
  jobs:
    orders_summary_job:
      name: orders-summary
      max_concurrent_runs: 1
      tasks:
        - task_key: build_orders_summary
          notebook_task:
            notebook_path: ../src/orders_summary.py
            base_parameters:
              catalog: ${var.catalog}
              schema: ${var.schema}
              run_environment: ${bundle.target}
          environment_key: default
      environments:
        - environment_key: default
          spec:
            client: '1'

## 9. Business notebook

The source notebook accepts target-specific parameters, creates a small country summary, writes a Delta table, and asserts the result. In a real project, keep notebook entry points thin and move reusable logic into tested Python modules. Never embed tokens, storage keys, passwords, workspace URLs, production catalog names, or personal paths in source code.

In [ ]:
# src/orders_summary.py - abbreviated; the companion project contains the full file.
dbutils.widgets.text('catalog', 'workspace')
dbutils.widgets.text('schema', 'bundle_demo')
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`')
# Transform and write a Delta summary table...

## 10. Validate before deployment

Validation checks bundle configuration and remote resource schemas. It does not prove that transformation logic is correct, input data exists, permissions are sufficient at runtime, or the deployed job will succeed. Run linting/unit tests separately, then validate every target.

In [ ]:
# From asset_bundle_example/
# databricks bundle validate -t dev --profile training
# databricks bundle validate -t prod --profile training
#
# Override a variable for one command:
# databricks bundle validate -t dev --var='catalog=main' --profile training

## 11. Deploy, inspect, and run

Deployment synchronizes selected files, creates or updates resources, and records deployment state for the bundle identity and target. Re-running the same deployment is expected to update the managed resource rather than create a second copy. Changing bundle identity, target, root/state paths, or deployment identity without planning can create ownership/state conflicts.

In [ ]:
# databricks bundle deploy -t dev --profile training
# databricks bundle summary -t dev --profile training
# databricks bundle run -t dev orders_summary_job --profile training
# databricks bundle run -t dev orders_summary_job --params catalog=workspace,schema=my_demo --profile training
#
# Open a generated resource URL from the summary, then inspect the run output and table.

### Deployment identity versus run identity

The **deployment identity** authenticates the CLI and creates/updates resources. The target's **`run_as` identity** executes the deployed workflow. Production normally uses a CI service principal for both roles or separate least-privilege principals where separation of duties requires it. The run identity needs data and compute permissions; the deployment identity needs workspace/resource management permissions. Do not rely on a developer's personal identity in production.

## 12. GitHub integration: two different features

1. **Databricks Git folders** clone/synchronize source code into a workspace for interactive collaboration. They version code files, but by themselves do not source-control all job/pipeline configuration.
2. **GitHub Actions + bundles** validate and deploy code and resource configuration from CI. This is the recommended production automation pattern.

Teams can use both: develop through local IDEs or Git folders, review changes in GitHub pull requests, and let GitHub Actions deploy the bundle after merge. Avoid manually editing a production resource that the bundle manages because the next deployment can replace the drift.

## 13. GitHub repository workflow

Recommended branch flow:

1. Developer creates a feature branch.
2. Developer validates/deploys their personal `dev` target.
3. Pull request runs lint, unit tests, and `bundle validate`.
4. Reviewer checks code plus resource, permission, schedule, and cost changes.
5. Merge to protected `main` triggers production validation and deployment.
6. GitHub `production` Environment can require approval and restrict deployment branches.
7. CI runs a smoke test or deployed job, publishes evidence, and alerts on failure.

Pin third-party actions to trusted versions or immutable commit SHAs according to organizational supply-chain policy. Protect `CODEOWNERS` paths for production bundle and workflow files.

## 14. Secretless GitHub authentication with OIDC

Databricks recommends workload identity federation for automated workloads. GitHub issues a short-lived OIDC identity token; Databricks validates it against a federation policy and exchanges it for Databricks OAuth access. No long-lived Databricks token is stored in GitHub.

One-time administrator setup:

1. Create a Databricks service principal.
2. Add it to the workspace and grant only required deployment/run/data permissions.
3. Create a federation policy for issuer `https://token.actions.githubusercontent.com`.
4. Restrict the subject to the repository and preferably a protected GitHub Environment, for example `repo:ORG/REPO:environment:production`.
5. Set the audience expected by the policy, commonly the GitHub organization URL.
6. Add GitHub variables `DATABRICKS_HOST` and `DATABRICKS_CLIENT_ID`. These are identifiers, not secrets.
7. Grant the workflow `id-token: write` and set `DATABRICKS_AUTH_TYPE=github-oidc`.

Do not use a subject broad enough to trust every repository or branch in an organization. Create separate policies/identities for materially different environments when appropriate.

In [ ]:
# Essential GitHub Actions authentication configuration (YAML)
permissions:
  contents: read
  id-token: write
env:
  DATABRICKS_HOST: ${{ vars.DATABRICKS_HOST }}
  DATABRICKS_CLIENT_ID: ${{ vars.DATABRICKS_CLIENT_ID }}
  DATABRICKS_AUTH_TYPE: github-oidc

## 15. Complete GitHub Actions behavior

The companion `.github/workflows/databricks-bundle.yml` performs:

- Checkout and CLI installation.
- Identity verification with `databricks current-user me`.
- Development-target validation for pull requests and pushes.
- Production deployment only on a push to `main`.
- Protected GitHub Environment `production`.
- Concurrency control so two production deployments do not overlap.
- `bundle run` as a simple post-deployment smoke test.

For a larger project, add Python/SQL linting, unit tests, bundle schema checks for every target, policy checks, artifact build/signing, integration tests in staging, approval, production deployment, smoke tests, and rollback/runbook evidence.

In [ ]:
# Core CI commands from the completed workflow
# databricks current-user me
# databricks bundle validate -t prod
# databricks bundle deploy -t prod --auto-approve
# databricks bundle run -t prod orders_summary_job

## 16. Production configuration checklist

Before deploying this example to production, replace all placeholders and verify:

- Exact production workspace host.
- Databricks service-principal application/client ID.
- GitHub federation issuer, audience, and environment-specific subject.
- GitHub Environment protection and repository variables.
- Deployment identity can manage bundle resources and workspace paths.
- `run_as` identity can use required compute and Unity Catalog objects.
- Production root/state paths are stable and not user-specific.
- Permissions name real groups/principals and follow least privilege.
- Output catalog/schema and ownership are approved.
- Schedules are intentionally paused/unpaused.
- Cost tags, timeout, retry, notification, and concurrency policies are present.

## 17. Variable precedence and safe configuration

Variables can have defaults, target overrides, command-line overrides, and lookup-backed values. Use variables for deploy-time differences, not secrets. Secrets should remain in Databricks secret scopes or a cloud secret manager and be referenced at runtime.

Examples:

```bash
databricks bundle deploy -t dev --var='catalog=main,schema=orders_dev'
databricks bundle validate -t prod
```

Prefer committed target configuration for production so the reviewed file describes what will deploy. Use command-line overrides mainly for controlled development or an explicitly governed release process.

## 18. Artifacts, libraries, and tests

For production Python projects, package reusable transformations as a wheel:

```yaml
artifacts:
  default:
    type: whl
    path: .
resources:
  jobs:
    example:
      tasks:
        - task_key: run
          python_wheel_task:
            package_name: my_project
            entry_point: main
          libraries:
            - whl: ./dist/*.whl
```

Run unit tests before `bundle deploy`. Bundle validation checks configuration, not business correctness. Pin dependencies and make artifact versions traceable to the Git commit.

## 19. Adopt an existing workspace job

Bundles can generate configuration for supported existing resources and bind bundle state to the existing object. This avoids recreating a manually built production job, but it must be treated as a controlled migration. Back up configuration, review generated YAML, validate it, bind to the exact resource ID, and deploy from one authoritative repository.

```bash
databricks bundle generate job --existing-job-id <job-id>
databricks bundle deployment bind <resource-key> <job-id> -t <target>
```

Check the current CLI help because supported generate/bind flags evolve. Do not bind a development target to a production job.

## 20. Cleanup and ownership

`bundle destroy` deletes resources managed by that deployment target after confirmation. It does not necessarily delete data tables written by jobs or external infrastructure. Before cleanup, confirm the active profile, workspace host, target, bundle identity, and summary. Never automate production destroy in an ordinary deployment workflow.

```bash
databricks bundle summary -t dev --profile training
databricks bundle destroy -t dev --profile training
```

## 21. Troubleshooting guide

| Symptom | Likely cause | Evidence/action |
|---|---|---|
| `bundle` command missing | Old/incorrect CLI on PATH | Check CLI version and executable path |
| Authentication succeeds locally but fails in Actions | Federation issuer/audience/subject mismatch or missing `id-token: write` | Compare GitHub token claims with policy; verify environment name |
| Validation fails on host/identity placeholder | Production placeholders not replaced | Set reviewed target values or CI environment variables |
| Deploy succeeds but job fails | Run identity lacks data/compute permission | Compare deployment identity with `run_as`; inspect run error |
| Duplicate resources | Bundle name, target, root/state path, or identity changed | Compare bundle summary and deployment state before further deploys |
| Deployment lock/state conflict | Concurrent deploys or different identities share state | Serialize CI; use stable deployment identity; investigate lock owner |
| File not found | Wrong relative path or sync exclusion | Validate path relative to its YAML file and inspect sync rules |
| Manual UI change disappears | Configuration drift overwritten by bundle | Make the change in source control and redeploy |
| Dev works, prod fails | Different host, identity, catalog, policy, or serverless availability | Validate prod explicitly and test staging with prod-like identity |
| OIDC works for PR unexpectedly | Federation policy subject too broad | Restrict subject to protected branch/environment |

## 22. Hands-on exercise

1. Copy `day10/asset_bundle_example` to a new GitHub repository or keep it within this repository.
2. Replace production placeholders.
3. Authenticate locally and validate `dev`.
4. Deploy and run the development job. Confirm the target table has three rows.
5. Change one input order, redeploy, rerun, and verify the update.
6. Open a pull request that modifies job tags and observe validation.
7. Configure a GitHub `production` Environment and Databricks OIDC federation.
8. Merge to `main`, approve production, and capture deployment/run evidence.
9. Demonstrate that no Databricks token exists in repository secrets.
10. Summarize the development target and intentionally destroy only that target.

### Acceptance criteria

- Both targets validate.
- Development resources are user-isolated.
- Production uses a service-principal `run_as` identity.
- GitHub uses OIDC with a repository/environment-restricted policy.
- Pull requests cannot deploy production.
- Deployment and smoke-test evidence links to the Git commit.
- No credentials are committed or stored as long-lived GitHub secrets.

## Official references

- [What are Declarative Automation Bundles?](https://docs.databricks.com/aws/en/dev-tools/bundles)
- [Bundle configuration reference](https://docs.databricks.com/aws/en/dev-tools/bundles/reference)
- [Bundle resources](https://docs.databricks.com/aws/en/dev-tools/bundles/resources)
- [Deployment modes](https://docs.databricks.com/aws/en/dev-tools/bundles/deployment-modes)
- [Bundle project templates](https://docs.databricks.com/aws/en/dev-tools/bundles/templates)
- [Databricks CLI authentication](https://docs.databricks.com/aws/en/dev-tools/cli/authentication)
- [GitHub Actions integration](https://docs.databricks.com/aws/en/dev-tools/ci-cd/github)
- [Workload identity federation for CI/CD](https://docs.databricks.com/aws/en/dev-tools/auth/oauth-federation-provider)
- [Configure a federation policy](https://docs.databricks.com/aws/en/dev-tools/auth/oauth-federation-policy)